In [ ]:
#| echo: false
import os

import retentioneering as rete

# Rendered by Quarto, slides have no Python kernel behind them, so every widget
# is baked in as a self-contained static page (render_static) instead of a live
# anywidget. Run in Jupyter, the widgets stay live.
if "QUARTO_DOCUMENT_PATH" in os.environ:
    from retentioneering.widgets._base import RetentioneeringWidget

    SLIDE_WIDGET_HEIGHT = 470

    def _static_bundle(self, include=None, exclude=None, **kwargs):
        return {"text/html": self.render_static(height=SLIDE_WIDGET_HEIGHT).data}

    RetentioneeringWidget._repr_mimebundle_ = _static_bundle

# Part 1: Introduction

## About us

### Vladimir Kukushkin

![](img/vladimir-kukushkin.jpg)

- 13 years in product analytics
- ex. Member of Techincal Staff @ Perplexity
- Principal AI Quality Analyst @ Finom
- Co-founder of Retentioneering

### Maxim Godzi

![](img/maxim-godzi.jpg)

- PhD in XXX
- NN years in product analytics
- ex. Principal Data Science Manager @ Microsoft
- Co-founder of Retentioneering

## UX research

| | Qualitative: why, how to fix | Quantitative: how many, how much |
|---|---|---|
| Behavioral: what people do | Moderated usability tests, Field studies, Diary studies | Event-log analysis ← we are here, A/B tests, Usability benchmarks |
| Attitudinal: what people say | Interviews, Focus groups, Open-ended feedback | Surveys, Card sorting, Feedback text mining |

<small>After C. Rohrer, "A Landscape of User Research Methods", Nielsen Norman Group</small>

## Event-log data

| user_id | session_id | event | timestamp | country | platform | n_session |
|---|---|---|---|---|---|---|
| user_0042 | sess_0042_1 | home | 2024-03-13 10:21:14 | US | desktop | 1 |
| user_0042 | sess_0042_1 | catalog | 2024-03-13 10:21:28 | US | desktop | 1 |
| user_0042 | sess_0042_1 | product_view | 2024-03-13 10:22:12 | US | desktop | 1 |
| user_0042 | sess_0042_2 | cart | 2024-03-15 19:02:40 | US | mobile | 2 |

- path identifiers: user_id, *session_id*
- event as a string: event
- event time: timestamp
- segments: *country*, *platform*
- miscellaneous: *n_session*

## Why event-log analysis is challenging

1. Number of event types
   - After cleanup and grouping: ~10
   - Small products: ~100
   - Medium products: a few hundred
   - Large products: 1,000+
2. Messy data
   - Same action, different names: `add_to_cart` (web), `AddToCart` (iOS), `cart_add` (Android)
   - Renamed without backfill: `checkout_start` until May, `begin_checkout` after
   - Raw URLs as events: `/product/18237`, `/product/90311?color=black`, … instead of `product_page`
3. Noisy patterns
   - A→B→C
   - A→X→B→C? (extra step)
   - A→B→B→C? (repeat)
   - B→A→C? (different order)
4. No "average" path
   - Users X (50%): A→B→C
   - Users Y (50%): D→B→E
   - Half of `B`s come from `A`, half end in `E`
   - Yet nobody went A→B→E.

# Part 2: Retentioneering tools

## The funnel tells you *where*, not *why*

In [ ]:
#| echo: false
stream = rete.datasets.load_ecom()

In [ ]:
stream.funnel(steps=["cart", "shipping_details", "purchase"], path_col="session_id")

::: {.notes}
Hook: 4.8% of sessions reach purchase. Ask the chat: what do the other 95% do instead?
:::

## An event log is all you need

In [ ]:
#| code-line-numbers: "|3"
import retentioneering as rete

stream = rete.datasets.load_ecom()
stream.df[["user_id", "session_id", "event", "timestamp", "platform"]].head(6)

. . .

Your own data: any table with **path id · event · timestamp** → `rete.Eventstream(df)`.

## The dataset has planted stories

A synthetic electronics store: 600 users, 3,605 sessions, 6 months.

| Planted story | When / who |
|---|---|
| Viral article: lots of browsing, few purchases | Feb 29 – Mar 3 |
| Checkout bug loops mobile users back to the cart | Apr 9 – May 9 |
| Payment gateway incident: errors and support chats | May 19 – Jun 7 |
| New users browse more, buy less than loyal ones | `user_lifecycle` |
| Cohort drift: older cohorts take direct paths | `user_cohort` |

. . .

Known ground truth → we can check whether a method **recovers** the story.

## One workflow, three scales

**Question** · **Shape events** · **Look** · **Compare** · **Check** · **Claim**

| The question sounds like… | Scale | Unit of analysis |
|---|---|---|
| "what happens right before / after X?" | **Transitions** | a pair of events |
| "how does a single visit unfold?" | **Sessions** | a session |
| "how do people get to value, over time?" | **Journeys** | a user |

. . .

- **Shape events:** data processors return a *new* eventstream (`add_segment`, `filter_paths`, `collapse_events`, …)
- **Look & compare:** every widget has a `diff=` mode: *A minus B*
- **Check:** every widget has a headless `*_data()` twin with the numbers

## The first look: a transition graph

In [ ]:
stream.transition_graph()

::: {.notes}
Live tour: node = event, edge = next-step probability. Search, focus a node, the sidebar.
Point: the graph generates hypotheses; it doesn't prove anything on its own.
:::

# Part 3: Use cases

## Scale 1 · Transitions

### A key metric dropped in late May. Where did journeys change?

## Compare *inside* vs *outside* the period

Question · **Shape events** · Look · **Compare** · Check · Claim

In [ ]:
#| code-line-numbers: 1|2
periods = stream.add_segment("period", time_range=("2024-05-19", "2024-06-07"))
periods.transition_graph(diff=("period", "inside", "outside"))

::: {.notes}
Prediction in chat before the reveal: which edges turn red?
Then click payment_details.
:::

## What follows an error?

In [ ]:
stream.step_sankey(anchor="payment_error", step_window=2, path_col="session_id")

## Check before you claim

> - How many paths are behind the red edge? → `transition_graph_data()`
> - Is the difference stable, or driven by a handful of users?
> - Is "inside" just a different mix of users (platform, channel)?

. . .

**Claim:** *inside the period, payment_details → payment_error → support_chat rises and → purchase falls.*
Not yet: *"the payment provider broke"*. That's a hypothesis for the engineering team.

## Your turn · 4 min

In the notebook, section **Scale 1**:

1. Move the period to **Apr 9 – May 9**
2. Compare it with the rest — what loop appears?
3. Add `path_col="session_id"`: does the picture change?

Post your one-sentence claim in the chat.

## Scale 2 · Sessions

### Where do sessions leave the checkout funnel, and what do they do instead?

## Stopped at shipping vs completed

In [ ]:
#| code-line-numbers: 1|3|4-9
steps = ["cart", "shipping_details", "purchase"]
(
    stream
        .add_segment("stage", funnel_events=steps, path_col="session_id")
        .step_matrix(
            path_pattern="cart->.*->shipping_details",
            step_window=2, path_col="session_id",
            diff=("stage", "shipping_details", "purchase"),
        )
)

## Sessions that match a behavior

Opened the cart, then left without reaching shipping or support:

In [ ]:
#| code-line-numbers: 1-7|8
abandoned = stream.filter_paths(
    {"metric": "matches_pattern", "op": "=", "value": True,
     "metric_args": {
         "pattern": "cart->[^shipping_details|support_chat]*->path_end"
     }},
    path_col="session_id",
)
abandoned.step_sankey(path_pattern="cart", path_col="session_id", step_window=3)

## Your turn · 4 min

TODO: session-level exercise (e.g. compare mobile vs desktop sessions in the funnel diff).

## Scale 3 · Journeys

### What kinds of users are there, and is that more than we already knew?

## Let behavior define the groups

In [ ]:
stream.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": "3-6"},
    overview_metrics=[
        {"metric": "event_count_bulk"},
        {"metric": "length"},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "user_lifecycle"}},
    ],
)

::: {.notes}
The key question: is a cluster a *strategy*, or just a known segment (new vs loyal) in disguise?
:::

## A cluster is not a persona

> - Does the cluster survive another `k`, another feature set, a subsample?
> - Is it just a known segment (`user_lifecycle`, `platform`) in disguise?
> - Can you name it from the heatmap — and does the name hold on the transition graph?

## How behavior changes across visits

In [ ]:
#| code-line-numbers: 3-6|7|8-13|14
visits = (
    stream
        .add_clusters(
            "session_type", features=[{"metric": "event_count_bulk"}],
            method_args={"n_clusters": 4}, path_col="session_id",
        )
        .collapse_events(group_col="session_id", name={"col": "session_type"})
        .rename_events({
            "cluster_0": "browsing",
            "cluster_1": "quick_visit",
            "cluster_2": "purchase_visit",
            "cluster_3": "promo_visit",
        })
)
visits.transition_graph()

## Your turn · 4 min

TODO: journey-level exercise (e.g. save clusters as a segment and diff them on the transition graph).

# Part 4 · Delegating to an AI agent

## Skill or MCP

TODO: two ways to hand the workflow to Claude Code / Codex; what stays your job (question, assumptions, validation, interpretation).

## Live demo · Google Merchandise Store

TODO: real GA4 data, no planted stories. Prompt card + what the agent must show (group sizes, filters, baseline).

## Reviewing what the agent did

TODO: validation checklist for agent output.

# Wrap-up

## Question → scale → evidence

TODO: framework recap + feature map (question → tool → where we met it).

## Bring your own data

TODO: GA4 / Amplitude / Mixpanel mapping, links, notebook.